# Long-budget learning order — analysis

Compare the **9,000-update constant versus cosine learning-rate studies** using the same five seeds and four curricula. This analysis separates the effects of *task ordering* from the effects of *optimization time* without training a new model.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path().cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BASE = PROJECT_ROOT / 'outputs/medium_world/experiments'
paths = {
    'constant': BASE / 'long_budget_constant/schedule_comparison.json',
    'cosine': BASE / 'long_budget_cosine_seed42/schedule_comparison.json'
}
reports = {name: json.loads(path.read_text(encoding='utf-8'))
           for name, path in paths.items()}
SCHEDULES = ('mixed', 'staged', 'staged_rehearsal', 'reverse')
TASKS = ('color', 'shape', 'conjunction', 'pair')
print({name: (len(report['runs']), sorted({r['seed'] for r in report['runs']}))
       for name, report in reports.items()})

## 1. What is controlled?

Each model receives exactly **45 presentations per training example**, totaling 450,000 presentations and 9,000 updates. The cosine directory name ends with `seed42` for historical reasons, but its JSON records **five seeds**. Schedule comparisons within one LR policy control the learning-rate trajectory; constant-versus-cosine comparisons test a separate intervention.

In [ ]:
protocol = pd.DataFrame([{
    'lr_policy': name, 'seed': r['seed'], 'schedule': r['schedule'],
    'updates': r['steps_run'],
    'total_presentations': r['phase_plan']['total_presentations'],
    'per_example': r['phase_plan']['repetitions_per_example'],
    **r['phase_plan']['counts_by_template']
} for name, report in reports.items() for r in report['runs']])
display(protocol.groupby(['lr_policy', 'schedule'])
        [['updates', 'total_presentations', 'per_example']].agg(['min', 'max']))

## 2. Final accuracy, OOD transfer, and binding efficiency

The main outcome is final ID **same-object binding**. OOD accuracy and OOD cross-entropy test unseen combinations. Binding AULC summarizes the entire learning trajectory, including the period before staged curricula begin binding training.

In [ ]:
rows = []
for name, report in reports.items():
    for r in report['runs']:
        f = r['final']
        rows.append({
            'lr_policy': name, 'seed': r['seed'], 'schedule': r['schedule'],
            'overall_id': 100 * f['id_dev']['accuracy'],
            'ood_pair': 100 * f['ood_dev']['by_template']['pair'],
            'ood_bce': f['ood_dev']['loss'],
            'pair_aulc': 100 * r['id_pair_aulc'],
            **{task: 100 * f['id_dev']['by_template'][task] for task in TASKS}
        })
final = pd.DataFrame(rows)
metrics = ['overall_id', 'pair', 'shape', 'ood_pair', 'ood_bce', 'pair_aulc']
display(final.groupby(['lr_policy', 'schedule'])[metrics]
        .agg(['mean', 'std']).round(2))

## 3. Paired effects of schedule and learning rate

For a seed-matched comparison, \(\Delta_s=A_{s,1}-A_{s,2}\). We compare curricula under the **same LR policy** and then compare constant versus cosine **for each curriculum**. Five-seed means and wins are descriptive, not conclusive statistical evidence.

In [ ]:
comparisons = (('staged_rehearsal', 'staged'), ('staged_rehearsal', 'mixed'),
               ('staged', 'mixed'), ('reverse', 'mixed'))
results = []
for lr_policy in reports:
    part = final[final.lr_policy == lr_policy]
    for first, second in comparisons:
        for metric in ('pair', 'pair_aulc', 'ood_pair', 'overall_id'):
            pivot = part.pivot(index='seed', columns='schedule', values=metric)
            delta = pivot[first] - pivot[second]
            results.append({'lr_policy': lr_policy, 'contrast': first + ' − ' + second,
                            'metric': metric, 'mean_pp': delta.mean(),
                            'sd_pp': delta.std(), 'wins': (delta > 0).sum()})
display(pd.DataFrame(results).round(2))

In [ ]:
cross_lr = []
for schedule in SCHEDULES:
    part = final[final.schedule == schedule]
    for metric in ('pair', 'pair_aulc', 'ood_pair', 'overall_id', 'ood_bce'):
        pivot = part.pivot(index='seed', columns='lr_policy', values=metric)
        delta = pivot['constant'] - pivot['cosine']
        cross_lr.append({'schedule': schedule, 'metric': metric,
                         'constant_minus_cosine': delta.mean(),
                         'wins': (delta > 0).sum()})
display(pd.DataFrame(cross_lr).round(3))

## 4. Learning curves

For binding accuracy \(A(t)\), the area under the learning curve is

\[\mathrm{AULC}=\frac{1}{T}\int_0^T A(t)\,dt.\]

It measures performance averaged over **all optimizer updates**. It should not be interpreted as improvement per binding example, because staged and mixed schedules introduce the task at different times.

In [ ]:
history = pd.DataFrame([{
    'lr_policy': name, 'seed': r['seed'], 'schedule': r['schedule'],
    'step': h['step'], 'phase': h['phase'], 'lr': h['lr'],
    **{task: 100 * h['id_dev']['by_template'][task] for task in TASKS}
} for name, report in reports.items() for r in report['runs'] for h in r['history']])
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, lr_policy in zip(axes, reports):
    part = history[history.lr_policy == lr_policy]
    for schedule in SCHEDULES:
        curve = part[part.schedule == schedule].groupby('step')['pair'].agg(['mean', 'std'])
        ax.plot(curve.index, curve['mean'], label=schedule)
        ax.fill_between(curve.index, curve['mean'] - curve['std'],
                        curve['mean'] + curve['std'], alpha=.10)
    ax.set(title=lr_policy.title() + ' LR', xlabel='Optimizer updates')
    ax.legend()
axes[0].set_ylabel('ID binding accuracy (%)')
plt.tight_layout()
plt.show()

## 5. Why constant and cosine update differently

Both studies use AdamW. The difference is the *externally chosen step size*. Constant LR holds \(\eta_t=10^{-4}\); cosine LR follows

\[\eta_t=\eta_{\min}+\frac{\eta_{\max}-\eta_{\min}}{2}
\left[1+\cos\left(\pi\frac{t-1}{T-1}\right)\right].\]

Here \(\eta_{\max}=10^{-4}\), \(\eta_{\min}=10^{-5}\), and \(T=9000\). The earlier ReduceLROnPlateau experiments lowered LR in response to validation loss; this cosine schedule does **not** react to validation and does **not** reset at stage changes.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
for name in reports:
    series = history[(history.lr_policy == name) &
                     (history.seed == 42) & (history.schedule == 'mixed')]
    ax.plot(series.step, series.lr, label=name)
ax.set(title='Learning-rate trajectory', xlabel='Optimizer updates', ylabel='LR')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Retention when task mixtures change

A transition may temporarily reduce accuracy on earlier skills. We compare consecutive evaluation checkpoints that straddle each stage change (normally 100 updates apart). These descriptive changes combine a new task mixture with continued optimization.

In [ ]:
changes = []
for name, report in reports.items():
    for r in report['runs']:
        for before, after in zip(r['history'], r['history'][1:]):
            if before['phase'] == after['phase']:
                continue
            changes.append({
                'lr_policy': name, 'schedule': r['schedule'], 'seed': r['seed'],
                'transition': before['phase'] + ' → ' + after['phase'],
                **{task: 100 * (after['id_dev']['by_template'][task] -
                                before['id_dev']['by_template'][task]) for task in TASKS}
            })
changes = pd.DataFrame(changes)
display(changes.groupby(['lr_policy', 'schedule', 'transition'])
        [list(TASKS)].mean().round(2))

## How to interpret this phase

Distinguish **final binding**, **learning efficiency**, and **OOD generalization**. If the curriculum ranking changes between constant and cosine LR, the optimizer interacts with task order. It does not yet establish that the ViT learned reusable object-level representations. The OOD development set was investigated repeatedly and is not a final, independent test.